# SQL drills — graduated, 18 problems

Each drill trains **one** pattern and builds on the one before it. Work top to bottom; skipping
ahead won't work, because D13–D18 assume the earlier ones are automatic.

**Total: ~95 minutes if you already know the material. Budget 2–3× that the first time through.**

| | drills | patterns |
|---|---|---|
| **Warm-up** | D01–D04 | `WHERE` · `GROUP BY` · `HAVING` · the integer-division trap |
| **Joins** | D05–D08 | two-table · three-table · anti-join · conditional aggregation |
| **Working with data** | D09–D12 | dates · `COUNT(DISTINCT)` · subqueries · CTEs |
| **Windows** | D13–D18 | `ROW_NUMBER` · `RANK` · `LAG` · `LEAD` · running totals · gaps-and-islands |

### How to use these — this matters more than the problems

For anything you can't write cold in 2 minutes:

1. **Attempt for 2 minutes.** Write whatever you've got, even if it's broken.
2. **Read the solution once.** Don't type while looking.
3. **Break it.** Delete a clause, predict what changes, run it, check. Three or four per query.
4. **Close the solution and write it from memory.** Stuck for 60 seconds? Peek at *one line*,
   close it, start over from the top.
5. **Repeat 4 until it's clean with no peeking.**
6. **Next day, write it again cold.**

Step 6 does more than steps 1–5 combined. One spaced rewrite beats three same-day repetitions.

The reference solutions are in `DRILLS_SOLUTIONS.md` — same rule as the exam, don't open it until
you've attempted.

In [1]:
import os, sys, sqlite3, time
import pandas as pd

SET_NAME = "capital-one-ds"

def _find_set_root(name):
    start = os.path.abspath("")
    p = start
    while True:
        if all(os.path.exists(os.path.join(p, f)) for f in ("grader.py", "make_key.py")):
            return p
        parent = os.path.dirname(p)
        if parent == p: break
        p = parent
    p = start
    while True:
        cand = os.path.join(p, "sets", name)
        if os.path.isdir(cand): return cand
        parent = os.path.dirname(p)
        if parent == p: break
        p = parent
    raise RuntimeError(f"Could not find the '{name}' problem set.")

SET_ROOT = _find_set_root(SET_NAME)
DATA = os.path.join(SET_ROOT, "data")
if SET_ROOT not in sys.path:
    sys.path.insert(0, SET_ROOT)
if not os.path.isdir(DATA):
    raise RuntimeError(f"No data yet. Run:  {os.path.join(SET_ROOT, 'bootstrap.sh')}")

CON = sqlite3.connect(os.path.join(DATA, "warehouse.db"))

def q(sql):
    if not sql.strip():
        print("(empty query -- write your SQL between the triple quotes)")
        return pd.DataFrame()
    return pd.read_sql_query(sql, CON)

from grader import check, score

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

print("tables:", q("SELECT name FROM sqlite_master WHERE type='table'").name.tolist())
print("\nschema reminder:")
for t in ["customers", "accounts", "transactions", "statements"]:
    cols = q(f"PRAGMA table_info({t})").name.tolist()
    print(f"  {t:13s} {', '.join(cols)}")

tables: ['customers', 'accounts', 'transactions', 'statements']

schema reminder:
  customers     customer_id, signup_date, state, age, segment, annual_income, credit_score_at_signup
  accounts      account_id, customer_id, product, open_date, credit_limit, apr, status
  transactions  transaction_id, account_id, txn_ts, amount, merchant_category, merchant_id, channel, is_disputed
  statements    statement_id, account_id, statement_month, statement_balance, min_payment_due, payment_made, days_past_due


---
## Warm-up — `WHERE`, `GROUP BY`, `HAVING`

### D01 — SELECT / WHERE / ORDER BY / LIMIT  ·  ~2 min

Return the 10 **Venture** accounts with a credit limit of **2600 or more**.

Columns: `account_id, product, credit_limit`. Order by `credit_limit` descending, then
`account_id` ascending. Cap the output at 10 rows.

In [2]:
# --- D01 ---
d01 = q("""
SELECT account_id, product, credit_limit
FROM accounts
WHERE product = "Venture" AND credit_limit >= 2600
ORDER BY credit_limit DESC, account_id
LIMIT 10
""")
check("D01", d01)

--- D01 ---------------------------------------------------------
PASS D01: 10 rows, all values match
    D01 score: 100%



### D02 — COUNT / AVG with GROUP BY  ·  ~3 min

One row per card product: how many accounts, the average credit limit, and the average APR.

Columns: `product, n_accounts, avg_limit, avg_apr`. Both averages rounded to 2dp.
Order by `n_accounts` descending.

In [4]:
# --- D02 ---
d02 = q("""
SELECT product, COUNT(*) AS n_accounts, ROUND(AVG(credit_limit), 2) AS avg_limit, ROUND(AVG(apr), 2) AS avg_apr
FROM accounts
GROUP BY product
ORDER BY n_accounts DESC
""")
check("D02", d02)

--- D02 ---------------------------------------------------------
PASS D02: 4 rows, all values match
    D02 score: 100%



### D03 — GROUP BY + HAVING  ·  ~3 min

States with **at least 250 customers**.

Columns: `state, n_customers`. Order by `n_customers` descending, then `state` ascending.

> The filter is on a group total, not on a row. That decides which keyword you need.

In [6]:
# --- D03 ---
d03 = q("""
SELECT state, COUNT(*) AS n_customers
FROM customers
GROUP BY state
HAVING n_customers >= 250
ORDER BY n_customers DESC
""")
check("D03", d03)

--- D03 ---------------------------------------------------------
PASS D03: 5 rows, all values match
    D03 score: 100%



### D04 — integer division -- the 1.0 * trap  ·  ~4 min

Per segment: how many customers, how many are missing `annual_income`, and what share
that is.

Columns: `segment, n_customers, n_missing_income, pct_missing`. `pct_missing` rounded to 4dp.
Order by `pct_missing` descending, then `segment` ascending.

> If your percentages all come out as 0, you've hit the single most common SQL bug there is.

In [7]:
# --- D04 ---
d04 = q("""
SELECT segment, COUNT(*) AS n_customers, SUM(CASE WHEN annual_income IS NULL THEN 1 ELSE 0 END) AS n_missing_income, ROUND(1.0 * SUM(CASE WHEN annual_income IS NULL THEN 1 ELSE 0 END) / COUNT(*), 4) AS pct_missing
FROM customers
GROUP BY segment
ORDER BY pct_missing DESC, segment
""")
check("D04", d04)

--- D04 ---------------------------------------------------------
PASS D04: 4 rows, all values match
    D04 score: 100%



---
## Joins

### D05 — two-table JOIN + GROUP BY  ·  ~4 min

Account counts and average credit limit **by customer segment** — the segment lives on
`customers`, the limit on `accounts`.

Columns: `segment, n_accounts, avg_limit`. `avg_limit` rounded to 2dp.
Order by `avg_limit` descending.

In [8]:
# --- D05 ---
d05 = q("""
SELECT c.segment, COUNT(*) AS n_accounts, ROUND(AVG(a.credit_limit), 2) AS avg_limit
FROM customers c
JOIN accounts a ON a.customer_id = c.customer_id
GROUP BY c.segment
ORDER BY avg_limit DESC
""")
check("D05", d05)

--- D05 ---------------------------------------------------------
PASS D05: 4 rows, all values match
    D05 score: 100%



### D06 — three-table JOIN  ·  ~5 min

Purchase counts and totals by customer segment **and** transaction channel. Purchases only
(`amount > 0`).

Columns: `segment, channel, n_txns, total_amount`. `total_amount` rounded to 2dp.
Order by `segment` ascending, then `channel` ascending.

> Three tables. Chain the joins: transactions → accounts → customers.

In [11]:
# --- D06 ---
d06 = q("""
SELECT c.segment, t.channel, COUNT(*) AS n_txns, ROUND(SUM(t.amount), 2) AS total_amount
FROM customers c
JOIN accounts a ON a.customer_id = c.customer_id
JOIN transactions t ON t.account_id = a.account_id
WHERE t.amount > 0
GROUP BY c.segment, t.channel
ORDER BY c.segment, t.channel
""")
check("D06", d06)

--- D06 ---------------------------------------------------------
PASS D06: 8 rows, all values match
    D06 score: 100%



### D07 — LEFT JOIN + IS NULL (anti-join)  ·  ~6 min

Per segment, how many customers have **never** had a delinquent statement
(`days_past_due > 0` on any of their accounts, in any month).

Columns: `segment, n_never_delinquent`. Order by `n_never_delinquent` descending,
then `segment` ascending.

> This is an **anti-join**: rows on the left with no match on the right. Build the set of
> delinquent customers first, LEFT JOIN to it, and keep the rows where the match is NULL.
> Careful — joining `statements` directly to `customers` fans out (one row per statement) and
> inflates every count. Collapse to one row per customer before you join.

In [16]:
# --- D07 ---
d07 = q("""
SELECT c.segment, COUNT(*) AS n_never_delinquent
FROM customers c
LEFT JOIN(
    SELECT a.customer_id, s.days_past_due
    FROM accounts a
    JOIN  statements s ON a.account_id = s.account_id
    WHERE s.days_past_due > 0
) d ON c.customer_id = d.customer_id
WHERE d.customer_id IS NULL
GROUP BY c.segment
ORDER BY n_never_delinquent DESC, segment
""")
check("D07", d07)

--- D07 ---------------------------------------------------------
PASS D07: 4 rows, all values match
    D07 score: 100%



### D08 — conditional aggregation with CASE WHEN  ·  ~5 min

One row per product, with account status broken out into columns.

Columns: `product, n_accounts, n_open, n_closed, n_charged_off`. Order by `product` ascending.

> Pivoting rows into columns. `SUM(CASE WHEN ... THEN 1 ELSE 0 END)` — this is the single most
> reusable aggregation idiom in SQL.

In [17]:
# --- D08 ---
d08 = q("""
SELECT product, COUNT(*) AS n_accounts, SUM(CASE WHEN status = "open" THEN 1 ELSE 0 END) AS n_open, SUM(CASE WHEN status = "closed" THEN 1 ELSE 0 END) AS n_closed, SUM(CASE WHEN status = "charged_off" THEN 1 ELSE 0 END) AS n_charged_off
FROM accounts
GROUP BY product
ORDER BY product
""")
check("D08", d08)

--- D08 ---------------------------------------------------------
PASS D08: 4 rows, all values match
    D08 score: 100%



---
## Dates, distinct counts, subqueries, CTEs

### D09 — date handling with strftime  ·  ~4 min

Monthly purchase volume for 2025 (`amount > 0`).

Columns: `month` formatted `'YYYY-MM'`, `n_txns`, `total_amount` rounded to 2dp.
Order by `month` ascending.

> `strftime('%Y-%m', txn_ts)` in SQLite.

In [21]:
# --- D09 ---
d09 = q("""
SELECT strftime('%Y-%m', txn_ts) AS month, COUNT(*) AS n_txns, ROUND(SUM(amount), 2) AS total_amount
FROM transactions
WHERE amount > 0
GROUP BY month
ORDER BY month
""")
check("D09", d09)

--- D09 ---------------------------------------------------------
PASS D09: 12 rows, all values match
    D09 score: 100%



### D10 — COUNT(DISTINCT ...)  ·  ~4 min

Per merchant category (non-null), the number of purchases, the number of **distinct
accounts**, and the number of **distinct merchants**.

Columns: `merchant_category, n_txns, n_accounts, n_merchants`.
Order by `n_txns` descending, then `merchant_category` ascending.

In [24]:
# --- D10 ---
d10 = q("""
SELECT merchant_category, COUNT(*) AS n_txns, COUNT(DISTINCT account_id) AS n_accounts, COUNT(DISTINCT merchant_id) AS n_merchants
FROM transactions
WHERE merchant_category IS NOT NULL AND amount > 0
GROUP BY merchant_category
ORDER BY n_txns DESC, merchant_category
""") 
check("D10", d10)

--- D10 ---------------------------------------------------------
PASS D10: 12 rows, all values match
    D10 score: 100%



### D11 — subquery in WHERE  ·  ~5 min

Customers whose `credit_score_at_signup` is **above the overall average**, counted by segment.

Columns: `segment, n_customers`. Order by `n_customers` descending, then `segment` ascending.

> The average is a single value, so a scalar subquery in the WHERE clause does it.

In [27]:
# --- D11 ---
d11 = q("""
SELECT segment, COUNT(*) AS n_customers
FROM customers
WHERE credit_score_at_signup >= (SELECT AVG(credit_score_at_signup) FROM customers)
GROUP BY segment
ORDER BY n_customers DESC, segment
""")
check("D11", d11)

--- D11 ---------------------------------------------------------
PASS D11: 4 rows, all values match
    D11 score: 100%



### D12 — CTE (WITH ...)  ·  ~5 min

Average total 2025 purchase spend per account, by product.

Columns: `product, n_accounts, avg_spend`. `avg_spend` rounded to 2dp.
Order by `avg_spend` descending.

> Two levels of aggregation: sum to the account first, then average those sums. A CTE makes this
> readable — and this shape (aggregate, then aggregate again) is most of real analytics SQL.

In [28]:
# --- D12 ---
d12 = q("""
WITH acc_spend AS(
    SELECT account_id, SUM(amount) AS tot_spend
    FROM transactions
    WHERE amount > 0
    GROUP BY account_id
)

SELECT a.product, COUNT(*) AS n_accounts, ROUND(AVG(s.tot_spend), 2) AS avg_spend
FROM accounts a
JOIN acc_spend s ON a.account_id = s.account_id
GROUP BY a.product
ORDER BY avg_spend desc
""")
check("D12", d12)

--- D12 ---------------------------------------------------------
PASS D12: 4 rows, all values match
    D12 score: 100%



---
## Window functions

### D13 — ROW_NUMBER -- top N per group  ·  ~7 min

The **3 highest-limit accounts within each product**.

Columns: `product, account_id, credit_limit, rn` (the rank, 1–3). Ties broken by `account_id`
ascending. Order by `product` ascending, then `rn` ascending.

> The top-N-per-group pattern. `ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)` in a CTE,
> then filter on the rank. **Memorize this one** — it appears in almost every SQL screen.

In [33]:
# --- D13 ---
d13 = q("""
WITH ranked AS(
    SELECT account_id, product, credit_limit,
           ROW_NUMBER() OVER (PARTITION BY product ORDER BY credit_limit DESC, account_id) AS rn
    FROM accounts
)

SELECT product, account_id, credit_limit, rn
FROM ranked
WHERE rn <= 3
ORDER BY product, rn
""")
check("D13", d13)

--- D13 ---------------------------------------------------------
PASS D13: 12 rows, all values match
    D13 score: 100%



### D14 — RANK vs ROW_NUMBER on ties  ·  ~6 min

Customer counts by state, with three different ranking functions side by side so you can see
how they differ on ties.

Columns: `state, n_customers, row_num, rnk, dense_rnk`.
- `row_num` = `ROW_NUMBER()` ordered by `n_customers` DESC then `state` ASC
- `rnk` = `RANK()` ordered by `n_customers` DESC only
- `dense_rnk` = `DENSE_RANK()` ordered by `n_customers` DESC only

Order by `n_customers` descending, then `state` ascending.

> Look hard at the output. Where two states tie, `RANK` repeats the number and then skips;
> `DENSE_RANK` repeats without skipping; `ROW_NUMBER` never repeats. Knowing which to reach for
> is a standard verbal question.

In [35]:
# --- D14 ---
d14 = q("""
WITH by_state AS (
    SELECT state, COUNT(*) AS n_customers
    FROM customers
    GROUP BY state
)

SELECT state, n_customers, 
       ROW_NUMBER() OVER (ORDER BY n_customers DESC, state) AS row_num,
       RANK() OVER (ORDER BY n_customers DESC) AS rnk,
       DENSE_RANK() OVER (ORDER BY n_customers DESC) AS dense_rnk
FROM by_state
ORDER BY n_customers DESC, state
""")
check("D14", d14)

--- D14 ---------------------------------------------------------
PASS D14: 15 rows, all values match
    D14 score: 100%



### D15 — LAG -- period over period  ·  ~7 min

Total payments received per statement month, with the prior month alongside and the change.

Columns: `month` (`'YYYY-MM'`), `total_paid` (2dp), `prev_paid`, `change` (2dp).
Order by `month` ascending. January's `prev_paid` and `change` are NULL.

> `LAG(col) OVER (ORDER BY month)`. Aggregate to one row per month **first** — a window function
> operates on result rows, so the months have to exist before you can lag across them.

In [ ]:
# --- D15 ---
d15 = q("""

""")
check("D15", d15)

--- D15 ---------------------------------------------------------
PASS D15: 12 rows, all values match
    D15 score: 100%



### D16 — LEAD with a next-period guard  ·  ~8 min

Of the accounts delinquent in a given month, how many **cured** (returned to
`days_past_due = 0`) the following calendar month?

Columns: `month` (`'YYYY-MM'`), `n_current` (accounts with `days_past_due > 0` that month),
`n_cured`. Cover 2025-01 through 2025-11. Order by `month` ascending.

> `LEAD` is `LAG` pointing forward. The catch: `LEAD` gives you the next *statement row*, which is
> only the next *calendar month* if the account's history has no gaps — so guard on
> `next_month = date(statement_month, '+1 month')`.

In [60]:
# --- D16 ---
d16 = q("""
WITH next AS(
    SELECT account_id, statement_month, days_past_due,
           LEAD(days_past_due) OVER (PARTITION BY account_id ORDER BY statement_month) AS next_dpd,
           LEAD(statement_month) OVER (PARTITION BY account_id ORDER BY statement_month) AS next_month
    FROM statements
)

SELECT strftime('%Y-%m', statement_month) AS month, COUNT(*) AS n_current, SUM(CASE WHEN next_dpd = 0 AND next_month = date(statement_month, '+1 month') THEN 1 ELSE 0 END) AS n_cured
FROM next
WHERE days_past_due > 0 AND statement_month < '2025-12-01'
GROUP BY month
ORDER BY month
""")
check("D16", d16)

--- D16 ---------------------------------------------------------
PASS D16: 11 rows, all values match
    D16 score: 100%



### D17 — running total with a window frame  ·  ~7 min

Monthly purchase spend for 2025 with a **cumulative running total**.

Columns: `month` (`'YYYY-MM'`), `spend` (2dp), `running_total` (2dp).
Order by `month` ascending.

> `SUM(spend) OVER (ORDER BY month ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)`.
> An `OVER` clause with an `ORDER BY` and a frame turns any aggregate into a running one.

In [62]:
# --- D17 ---
d17 = q("""
WITH monthly AS(
    SELECT strftime('%Y-%m', txn_ts) AS month, ROUND(SUM(amount), 2) AS spend
    FROM transactions
    WHERE amount > 0
    GROUP BY month
)

SELECT month, spend, ROUND(SUM(spend) OVER (ORDER BY month ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 2) AS running_total
FROM monthly
ORDER BY month

""")
check("D17", d17)

--- D17 ---------------------------------------------------------
PASS D17: 12 rows, all values match
    D17 score: 100%



### D18 — gaps and islands -- consecutive streaks  ·  ~10 min

For each account, find the **longest run of consecutive calendar months in which it had at
least one purchase**. Then report the distribution: how many accounts have each longest-streak value.

Columns: `longest_streak, n_accounts`. Order by `longest_streak` ascending.

> **Gaps and islands.** Number each account's active months with `ROW_NUMBER()`. Within a
> consecutive run, `month - row_number` is constant — so `date(month, '-' || rn || ' months')`
> gives one key per island. Group on it, count, then take each account's MAX.
>
> This is the capstone. If you can write it cold, S5 in the timed exam is the same problem.

In [65]:
# --- D18 ---
d18 = q("""
WITH entity_period AS(
    SELECT DISTINCT account_id, strftime('%Y-%m-01', txn_ts) AS month
    FROM transactions
    WHERE amount > 0
),
numbered AS(
    SELECT account_id, month, ROW_NUMBER() OVER (PARTITION BY account_id ORDER BY month) AS rn
    FROM entity_period
),
groups AS(
    SELECT account_id, date(month, '-' || rn || ' months') AS groups, COUNT(*) AS streak
    FROM numbered
    GROUP BY account_id, groups
)

SELECT streak AS longest_streak, COUNT(*) AS n_accounts
FROM (SELECT account_id, MAX(streak) AS streak FROM groups GROUP BY account_id)
GROUP BY streak
ORDER BY longest_streak
""")
check("D18", d18)

--- D18 ---------------------------------------------------------
PASS D18: 12 rows, all values match
    D18 score: 100%



---
## Done

```python
score()
```

Anything below 100% goes back through the read → break → close → rewrite loop, and gets rewritten
cold the next day.

When D13–D18 are automatic, re-sit **Section 2 of the timed exam** cold. That's the real test of
whether this transferred — the drills tell you the pattern, the exam makes you recognize it.

In [ ]:
score()